In [ ]:
import pandas as pd
import requests
import os
import yaml
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

In [2]:
yaml_file = 'mvc.yaml'

with open(yaml_file, 'r') as f:
    mvc_data = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
def get_data(base_url, mvc_subset, row_size=1000, datapage=1):

    url = base_url

    headers = {
        'X-App-Token': os.environ.get("X-APP-TOKEN")
    }

    payload = {
        'query': 'SELECT * ORDER BY crash_date ASC',
        'page': {
            'pageNumber': datapage,
            'pageSize': row_size
        }
    }

    basic = HTTPBasicAuth(os.environ.get("user"), os.environ.get("pass"))

    response = requests.post(url, json=payload, headers=headers, auth=basic)

    if response.status_code == 200:
        crashes_df = pd.json_normalize(response.json())
    else:
        return response.json()

    file_path =f'./sample_data/{mvc_subset}/pg_{datapage}_{mvc_subset}.csv'

    if not os.path.exists(file_path):
        crashes_df.to_csv(file_path, index=False, mode='a', header=True)
    else:
        crashes_df.to_csv(file_path, index=False, mode='a', header=False)

In [4]:
load_dotenv()
get_data(base_url=mvc_data.get("crashes_endpoint"),mvc_subset="crashes")

{'code': 'authentication_required',
 'error': True,
 'message': 'You must be logged in to access this resource'}